In [9]:
import pandas as pd
import numpy as np
from vis.utils_motifs import motif_to_fwords
from collections import Counter
import ast
import matplotlib.pyplot as plt
import torch
import math
from scipy.stats import ttest_rel

In [13]:
# file_name = r"results\run_GCN_persuade_wan_cooc_qwen2515b_wan-w5_nf-count+freq+emb_ew-wan_sp-random_L2_H64_sum_lr0.001_seed2026\edge_faithfulness_top0p050_100repeats.csv"
file_name = r"results\run_GCN_persuade_wan_cooc_qwen2515b_wan-w5_nf-count+freq+emb_ew-wan_sp-random_L2_H64_sum_lr0.001_seed2026\edge_faithfulness_top0p200_100repeats.csv"
df = pd.read_csv(file_name)
df

,row,class,original_confidence,top_removed_confidence,drop_top_removed,drop_random,random_drops,flip_top_removed,flip_random,random_flips
0,13,Human,0.999578,0.999600,-0.000022,-8.120716e-05,"[-7.706880569458008e-05, 0.0001580119132995605...",0,0.0,"[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ..."
1,17,Human,0.999645,0.999869,-0.000224,-1.324505e-04,"[-0.00026607513427734375, -0.00013035535812377...",0,0.0,"[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ..."
2,20,Human,0.999991,0.999980,0.000011,-9.667873e-07,"[-3.4570693969726562e-06, -3.337860107421875e-...",0,0.0,"[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ..."
3,22,Human,0.999956,0.999932,0.000024,-1.511931e-05,"[-2.5033950805664062e-05, 2.9802322387695312e-...",0,0.0,"[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ..."
4,28,Human,0.999999,0.999999,0.000000,-3.099442e-07,"[-4.76837158203125e-07, -5.960464477539062e-07...",0,0.0,"[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ..."
...,...,...,...,...,...,...,...,...,...,...
353,2372,Machine,0.988750,0.933135,0.055615,6.737338e-03,"[-0.0022145509719848633, -0.006129860877990723...",0,0.0,"[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ..."
354,2379,Machine,0.987462,0.763059,0.224403,3.062434e-02,"[-0.007053077220916748, 0.035359084606170654, ...",0,0.0,"[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ..."
355,2381,Machine,0.999972,0.998601,0.001371,5.373657e-05,"[4.303455352783203e-05, 5.8770179748535156e-05...",0,0.0,"[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ..."
356,2383,Machine,0.999962,0.997977,0.001986,6.362498e-05,"[-2.2292137145996094e-05, 1.1920928955078125e-...",0,0.0,"[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ..."


### Hypothesis Testing

Looking at these results, hypothesis is that there is not significant difference in the effect on the GNN's predictions between removing edges sleected by GNNExplainer and removing randomly selected edges. 

H_0 : no difference in mean prediction difference
H_1: there is a difference

In [14]:
for cls, group in df.groupby("class"):
    t, p = ttest_rel(
        group["drop_top_removed"],
        group["drop_random"],
        alternative="two-sided"
    )

    difference = group["drop_top_removed"] - group["drop_random"]

    print(f"\n{cls}")
    print(f"Mean top deletion: {group['drop_top_removed'].mean():.4f}")
    print(f"Mean random deletion: {group['drop_random'].mean():.4f}")
    print(f"Mean difference: {difference.mean():.4f}")
    print(f"p-value: {p:.6f}")


Human
Mean top deletion: -0.0050
Mean random deletion: -0.0010
Mean difference: -0.0039
p-value: 0.121091

Machine
Mean top deletion: 0.0997
Mean random deletion: 0.0063
Mean difference: 0.0935
p-value: 0.000000
